In [1]:
import pandas as pd

In [11]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [12]:
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int)

In [20]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1,4.07,6.82,34.12


In [14]:
row = df.iloc[0]
row

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                            1
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [15]:
from models import Ride, ride_from_row, ride_serializer

In [16]:
ride = ride_from_row(df.iloc[1])
ride
# we have an event, now how do we produce/stream it?

Ride(lpep_pickup_datetime=1759277643000, lpep_dropoff_datetime=1759278254000, PULocationID=66, DOLocationID=25, passenger_count=1, trip_distance=1.61, tip_amount=2.78, total_amount=16.68)

In [17]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer #this is a method of the class Ride, defined in models.py, it makes the ride into json and then a series of bytes
)

In [18]:
topic_name = 'green-trips'

In [19]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 2.21 seconds
